# SQL com Python — Cheat Sheet 🗄️

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vagnerx/excel-python-sql-powerbi-cheatsheet/blob/main/notebooks/sql_com_python.ipynb)

Neste notebook, exploramos como executar comandos SQL diretamente no Python usando `sqlite3` e `pandas`, conectando-se ao nosso banco de dados local `cheatsheet.db`.

In [ ]:
# ─── SETUP — Execute esta célula primeiro ────────────────────────────────────
import os, sys
import sqlite3
import pandas as pd

# Detecta se está no Colab e clona o repositório se necessário
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone https://github.com/vagnerx/excel-python-sql-powerbi-cheatsheet.git
    # Executa o script que gera o banco de dados a partir dos CSVs
    !python excel-python-sql-powerbi-cheatsheet/docs/examples/sql/create_db.py
    DB_PATH = 'excel-python-sql-powerbi-cheatsheet/datasets/cheatsheet.db'
    CSV_PATH = 'excel-python-sql-powerbi-cheatsheet/datasets/employees.csv'
else:
    # Execução local — ajusta path para a raiz do projeto
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if os.path.basename(os.getcwd()) == 'notebooks':
        REPO_ROOT = os.path.abspath('..')
    else:
        REPO_ROOT = os.getcwd()
    DB_PATH = os.path.join(REPO_ROOT, 'datasets', 'cheatsheet.db')
    CSV_PATH = os.path.join(REPO_ROOT, 'datasets', 'employees.csv')

# Conectando ao banco de dados
conn = sqlite3.connect(DB_PATH)
print("\nConectado ao SQLite com sucesso!")

### 1. Operações Básicas (Filtros e Selects)

In [ ]:
query = """
SELECT 
    nome, departamento, salario 
FROM employees 
WHERE salario > 8000 
ORDER BY salario DESC 
LIMIT 5;
"""
pd.read_sql(query, conn)

### 2. Agregações (GROUP BY, SUM, AVG)

In [ ]:
query = """
SELECT 
    departamento,
    COUNT(*) AS qtd_funcionarios,
    ROUND(AVG(salario), 2) AS media_salarial,
    SUM(salario) AS custo_total
FROM employees
GROUP BY departamento
ORDER BY custo_total DESC;
"""
pd.read_sql(query, conn)

### 3. Joins (Múltiplas Tabelas)

In [ ]:
query = """
SELECT 
    c.nome AS cliente,
    c.segmento,
    COUNT(o.id_pedido) AS total_pedidos,
    SUM(o.valor_total) AS gasto_total
FROM customers c
LEFT JOIN orders o ON c.id_cliente = o.id_cliente
GROUP BY c.id_cliente, c.nome, c.segmento
ORDER BY gasto_total DESC
LIMIT 5;
"""
pd.read_sql(query, conn)

### 4. Window Functions (Rank, Participação %)

In [ ]:
query = """
SELECT 
    nome, 
    departamento, 
    salario,
    RANK() OVER(PARTITION BY departamento ORDER BY salario DESC) AS rank_depto,
    ROUND((salario * 100.0) / SUM(salario) OVER(PARTITION BY departamento), 1) AS pct_do_depto
FROM employees
WHERE salario IS NOT NULL
LIMIT 10;
"""
pd.read_sql(query, conn)

### Alternativa: DuckDB (Sem Banco de Dados Pré-Criado)
O `duckdb` permite rodar SQL diretamente nos arquivos CSV, sem precisar popular um `.db` antes.

In [ ]:
# Descomente e instale se necessário: !pip install duckdb
import duckdb

query_duck = f"""
SELECT departamento, COUNT(*) as qtd
FROM read_csv_auto('{CSV_PATH}')
GROUP BY departamento
ORDER BY qtd DESC;
"""
duckdb.query(query_duck).df()

In [ ]:
# Fechando a conexão SQLite ao final
conn.close()